In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/kgan31/doha-context/dohas_nlp_ready_simple_lang.csv
/kaggle/input/datasets/kgan31/doha-simple-context/dohas_nlp_ready_simple_lang.csv


In [2]:
!pip install transformers sentencepiece datasets torch sacrebleu tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.8 MB/s eta 0:00:00


In [3]:
"""
IndicBART Fine-tuning for Hindi Doha Generation
================================================
Fine-tunes ai4bharat/IndicBART on (Theme + Context → Doha) pairs.
Includes tqdm progress bars with ETA for every stage:
  - Per-batch progress bar within each epoch (train + val)
  - Per-epoch overall progress bar with ETA for full training
  - BLEU evaluation progress bar

Kaggle setup:
    !pip install transformers sentencepiece datasets torch sacrebleu tqdm
    CSV_PATH = "/kaggle/input/doha-dataset/doha_dataset.csv"
    SAVE_DIR = "/kaggle/working"

Local setup:
    pip install transformers sentencepiece datasets torch sacrebleu tqdm pandas numpy
"""

import os
import re
import math
import random
import time
import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    get_linear_schedule_with_warmup,
)

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk
nltk.download("punkt", quiet=True)

from tqdm.auto import tqdm          # auto picks tqdm.notebook in Kaggle/Jupyter

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────

CSV_PATH      = "/kaggle/input/datasets/kgan31/doha-simple-context/dohas_nlp_ready_simple_lang.csv"       # ← change to your path
SAVE_DIR      = "/kaggle/working/"                      # ← where to save checkpoints

MODEL_NAME    = "ai4bharat/IndicBART"

EPOCHS        = 10
BATCH_SIZE    = 16
GRAD_ACCUM    = 2
LR            = 5e-5
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.1
MAX_GRAD_NORM = 1.0
VAL_SPLIT     = 0.1
PATIENCE      = 4

MAX_INPUT_LEN  = 64
MAX_TARGET_LEN = 128

NUM_BEAMS      = 4
GEN_MAX_LENGTH = 128
LANG_TOKEN     = "<2hi>"

SEED    = 42
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)


# ─────────────────────────────────────────────────────────────────────────────
# 1. DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = re.sub(r"।।\s*\d+\s*।।", "।।", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def format_input(theme: str, context: str) -> str:
    return f"विषय: {clean_text(theme)} संदर्भ: {clean_text(context)}"


def load_dataset(csv_path: str):
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]

    required = {"Doha", "Theme", "Context"}
    missing  = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}. Found: {list(df.columns)}")

    df = df.dropna(subset=["Doha", "Theme", "Context"]).reset_index(drop=True)

    pairs = []
    for _, row in df.iterrows():
        inp = format_input(str(row["Theme"]), str(row["Context"]))
        tgt = clean_text(str(row["Doha"]))
        if inp.strip() and tgt.strip():
            pairs.append((inp, tgt))

    print(f"✅ Loaded {len(pairs)} (input, doha) pairs")
    print(f"   Sample input  : {pairs[0][0]}")
    print(f"   Sample target : {pairs[0][1][:80]}...\n")
    return pairs, df


def split_pairs(pairs: list, val_split: float):
    data = pairs.copy()
    random.shuffle(data)
    split = int(len(data) * (1 - val_split))
    return data[:split], data[split:]


# ─────────────────────────────────────────────────────────────────────────────
# 2. DATASET
# ─────────────────────────────────────────────────────────────────────────────

class DohaDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_input_len, max_target_len, lang_token):
        self.pairs          = pairs
        self.tokenizer      = tokenizer
        self.max_input_len  = max_input_len
        self.max_target_len = max_target_len
        self.lang_token     = lang_token

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        inp, tgt = self.pairs[idx]

        enc = self.tokenizer(
            inp + " " + self.lang_token,
            max_length=self.max_input_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        # Fixed: removed deprecated as_target_tokenizer() context manager
        dec = self.tokenizer(
            tgt,
            max_length=self.max_target_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        labels = dec["input_ids"].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids":      enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "labels":         labels,
        }


# ─────────────────────────────────────────────────────────────────────────────
# 3. TRAINING  (with tqdm batch bars)
# ─────────────────────────────────────────────────────────────────────────────

def train_epoch(model, loader, optimizer, scheduler, device,
                grad_accum, max_grad_norm, epoch, total_epochs):
    """
    One training epoch with a tqdm batch-level bar showing:
      Epoch  3/20 [train] ██████░░ 142/506 | loss=1.823 | ETA 02:14
    """
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    bar = tqdm(
        loader,
        desc=f"Epoch {epoch:>2}/{total_epochs} [train]",
        unit="batch",
        leave=False,        # disappears when done, replaced by epoch summary
        dynamic_ncols=True,
    )

    for step, batch in enumerate(bar):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss / grad_accum
        loss.backward()

        if (step + 1) % grad_accum == 0 or (step + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += outputs.loss.item()
        bar.set_postfix(loss=f"{total_loss / (step + 1):.4f}")

    bar.close()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate_loss(model, loader, device, epoch, total_epochs):
    """
    Validation pass with its own tqdm bar showing:
      Epoch  3/20 [ val ] ████████ 57/57 | loss=2.341 | ETA 00:08
    """
    model.eval()
    total_loss = 0.0

    bar = tqdm(
        loader,
        desc=f"Epoch {epoch:>2}/{total_epochs} [ val ]",
        unit="batch",
        leave=False,
        dynamic_ncols=True,
    )

    for step, batch in enumerate(bar):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        total_loss += outputs.loss.item()
        bar.set_postfix(loss=f"{total_loss / (step + 1):.4f}")

    bar.close()
    return total_loss / len(loader)


# ─────────────────────────────────────────────────────────────────────────────
# 4. GENERATION
# ─────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def generate_doha(model, tokenizer, theme, context,
                  lang_token=LANG_TOKEN, num_beams=NUM_BEAMS,
                  max_length=GEN_MAX_LENGTH, device=DEVICE):
    model.eval()
    inp = format_input(theme, context) + " " + lang_token

    enc = tokenizer(
        inp,
        max_length=MAX_INPUT_LEN,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    forced_bos = tokenizer.convert_tokens_to_ids(lang_token)

    output_ids = model.generate(
        input_ids=enc["input_ids"],
        attention_mask=enc["attention_mask"],
        num_beams=num_beams,
        max_length=max_length,
        early_stopping=True,
        forced_bos_token_id=forced_bos if forced_bos != tokenizer.unk_token_id else None,
        no_repeat_ngram_size=3,
        length_penalty=1.0,
    )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()


# ─────────────────────────────────────────────────────────────────────────────
# 5. EVALUATION METRICS
# ─────────────────────────────────────────────────────────────────────────────

def compute_perplexity(avg_loss: float) -> float:
    return math.exp(min(avg_loss, 500))


def compute_bleu(reference_dohas: list, hypothesis: str) -> dict:
    smoother  = SmoothingFunction().method1
    hyp_chars = list(hypothesis)
    ref_chars = [list(r) for r in reference_dohas]
    scores    = {}
    for n in range(1, 5):
        weights = tuple([1.0 / n] * n + [0.0] * (4 - n))
        scores[f"bleu{n}"] = sentence_bleu(
            ref_chars, hyp_chars,
            weights=weights,
            smoothing_function=smoother
        )
    return scores


def run_bleu_evaluation(model, tokenizer, val_pairs, all_dohas,
                        n_samples=20, device=DEVICE):
    """BLEU evaluation with its own tqdm bar showing running BLEU-1."""
    samples   = random.sample(val_pairs, min(n_samples, len(val_pairs)))
    all_b1, all_b2, all_b4 = [], [], []

    bar = tqdm(
        samples,
        desc="  BLEU eval",
        unit="sample",
        dynamic_ncols=True,
    )

    for inp_text, _ in bar:
        m = re.match(r"विषय:\s*(.+?)\s*संदर्भ:\s*(.+)", inp_text)
        theme, context = "", ""
        if m:
            theme, context = m.group(1).strip(), m.group(2).strip()

        gen    = generate_doha(model, tokenizer, theme, context, device=device)
        scores = compute_bleu(all_dohas, gen)
        all_b1.append(scores["bleu1"])
        all_b2.append(scores["bleu2"])
        all_b4.append(scores["bleu4"])
        bar.set_postfix(bleu1=f"{np.mean(all_b1):.4f}")

    bar.close()

    results = {
        "bleu1": np.mean(all_b1),
        "bleu2": np.mean(all_b2),
        "bleu4": np.mean(all_b4),
    }
    print(f"   BLEU-1 : {results['bleu1']:.4f}")
    print(f"   BLEU-2 : {results['bleu2']:.4f}")
    print(f"   BLEU-4 : {results['bleu4']:.4f}")
    return results


# ─────────────────────────────────────────────────────────────────────────────
# 6. MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main():
    print(f"\n{'='*60}")
    print("IndicBART Fine-tuning — Hindi Doha Generation")
    print(f"Model  : {MODEL_NAME}")
    print(f"Device : {DEVICE}")
    print(f"{'='*60}\n")

    # ── Load data ──────────────────────────────────────────
    pairs, df = load_dataset(CSV_PATH)
    train_pairs, val_pairs = split_pairs(pairs, VAL_SPLIT)
    all_dohas = df["Doha"].apply(clean_text).tolist()

    print(f"   Train pairs : {len(train_pairs)}")
    print(f"   Val pairs   : {len(val_pairs)}\n")

    # ── Load model ─────────────────────────────────────────
    print(f"⬇  Loading {MODEL_NAME}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)

    total_params     = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   Total params     : {total_params:,}")
    print(f"   Trainable params : {trainable_params:,}\n")

    # ── Datasets & loaders ────────────────────────────────
    train_dataset = DohaDataset(train_pairs, tokenizer,
                                MAX_INPUT_LEN, MAX_TARGET_LEN, LANG_TOKEN)
    val_dataset   = DohaDataset(val_pairs, tokenizer,
                                MAX_INPUT_LEN, MAX_TARGET_LEN, LANG_TOKEN)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                              shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=0)

    # ── Optimizer & scheduler ─────────────────────────────
    optimizer    = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps  = (len(train_loader) // GRAD_ACCUM) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler    = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    print(f"   Total training steps : {total_steps}")
    print(f"   Warmup steps         : {warmup_steps}\n")

    # ── Outer epoch bar — tracks overall training + ETA ───
    epoch_bar = tqdm(
        range(1, EPOCHS + 1),
        desc="Overall progress",
        unit="epoch",
        dynamic_ncols=True,
    )

    best_val_loss      = float("inf")
    early_stop_counter = 0
    history            = []

    tqdm.write(
        f"\n{'Epoch':>6} {'Train Loss':>12} {'Val Loss':>10} "
        f"{'Train PPL':>11} {'Val PPL':>9} {'LR':>10} {'Time':>7}"
    )
    tqdm.write("-" * 68)

    for epoch in epoch_bar:
        t0 = time.time()

        train_loss = train_epoch(model, train_loader, optimizer, scheduler,
                                 DEVICE, GRAD_ACCUM, MAX_GRAD_NORM,
                                 epoch, EPOCHS)
        val_loss   = evaluate_loss(model, val_loader, DEVICE, epoch, EPOCHS)

        train_ppl  = compute_perplexity(train_loss)
        val_ppl    = compute_perplexity(val_loss)
        elapsed    = time.time() - t0
        current_lr = optimizer.param_groups[0]["lr"]

        history.append({
            "epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
            "train_ppl": train_ppl, "val_ppl": val_ppl, "lr": current_lr
        })

        # Update outer epoch bar with latest metrics
        epoch_bar.set_postfix(
            train_loss=f"{train_loss:.4f}",
            val_loss=f"{val_loss:.4f}",
            val_ppl=f"{val_ppl:.2f}",
        )

        # tqdm.write prints a persistent line that won't be overwritten by bars
        tqdm.write(
            f"{epoch:>6} {train_loss:>12.4f} {val_loss:>10.4f} "
            f"{train_ppl:>11.2f} {val_ppl:>9.2f} {current_lr:>10.2e} {elapsed:>6.1f}s"
        )

        # Save best checkpoint
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            early_stop_counter = 0
            ckpt_dir = os.path.join(SAVE_DIR, "best_indicbart_doha")
            os.makedirs(ckpt_dir, exist_ok=True)
            model.save_pretrained(ckpt_dir)
            tokenizer.save_pretrained(ckpt_dir)
            tqdm.write(f"         💾 Saved best model (val_ppl={val_ppl:.2f})")
        else:
            early_stop_counter += 1
            if early_stop_counter >= PATIENCE:
                tqdm.write(
                    f"\n⏹  Early stopping at epoch {epoch} "
                    f"(no improvement for {PATIENCE} epochs)"
                )
                break

        # Sample generation every 3 epochs
        if epoch % 3 == 0 and val_pairs:
            inp_text, ref = val_pairs[0]
            m = re.match(r"विषय:\s*(.+?)\s*संदर्भ:\s*(.+)", inp_text)
            if m:
                t, c = m.group(1).strip(), m.group(2).strip()
                gen  = generate_doha(model, tokenizer, t, c, device=DEVICE)
                tqdm.write(f"\n  🎵 Theme={t!r} | Context={c!r}")
                tqdm.write(f"     Generated → {gen}")
                tqdm.write(f"     Reference → {ref[:80]}\n")

    epoch_bar.close()

    # ── Final evaluation ───────────────────────────────────
    print(f"\n{'='*60}")
    print("FINAL EVALUATION — loading best checkpoint")
    print(f"{'='*60}\n")

    best_model = AutoModelForSeq2SeqLM.from_pretrained(
        os.path.join(SAVE_DIR, "best_indicbart_doha")
    ).to(DEVICE)
    best_tok   = AutoTokenizer.from_pretrained(
        os.path.join(SAVE_DIR, "best_indicbart_doha")
    )

    print("📊 BLEU evaluation on 20 samples...")
    bleu_scores = run_bleu_evaluation(
        best_model, best_tok, val_pairs,
        all_dohas, n_samples=20, device=DEVICE
    )

    history_df = pd.DataFrame(history)
    history_df.to_csv(os.path.join(SAVE_DIR, "indicbart_history.csv"), index=False)
    print(f"\n  📈 Training history saved → {SAVE_DIR}/indicbart_history.csv")

    # ── Summary ────────────────────────────────────────────
    print(f"\n{'='*60}")
    print("SUMMARY")
    print(f"{'='*60}")
    print(f"  Model        : {MODEL_NAME}")
    print(f"  Best Val PPL : {compute_perplexity(best_val_loss):.2f}")
    print(f"  BLEU-1       : {bleu_scores['bleu1']:.4f}")
    print(f"  BLEU-2       : {bleu_scores['bleu2']:.4f}")
    print(f"  BLEU-4       : {bleu_scores['bleu4']:.4f}")
    print(f"{'='*60}\n")

    # ── Generate final samples ─────────────────────────────
    print("GENERATED DOHAS (beam search, num_beams=4)")
    print("-" * 55)
    for inp_text, ref_doha in random.sample(val_pairs, min(5, len(val_pairs))):
        m = re.match(r"विषय:\s*(.+?)\s*संदर्भ:\s*(.+)", inp_text)
        if not m:
            continue
        t, c = m.group(1).strip(), m.group(2).strip()
        gen  = generate_doha(best_model, best_tok, t, c, device=DEVICE)
        print(f"  Theme     : {t}")
        print(f"  Context   : {c}")
        print(f"  Generated → {gen}")
        print(f"  Reference → {ref_doha[:100]}")
        print("-" * 55)


if __name__ == "__main__":
    main()


IndicBART Fine-tuning — Hindi Doha Generation
Model  : ai4bharat/IndicBART
Device : cuda

✅ Loaded 7889 (input, doha) pairs
   Sample input  : विषय: शृंगार संदर्भ: मोरपंखी बाल
   Sample target : मोर पच्छ जो सिर चढ़ै बारन तें अधिकाय। सहस चखन लखि धनि कचन परे मान छिन पाय ॥...

   Train pairs : 7100
   Val pairs   : 789

⬇  Loading ai4bharat/IndicBART...


config.json:   0%|          | 0.00/832 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.90M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/221 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/267 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


📊 BLEU evaluation on 20 samples...


  BLEU eval:   0%|          | 0/20 [00:00<?, ?sample/s]

   BLEU-1 : 0.9194
   BLEU-2 : 0.8043
   BLEU-4 : 0.5358

  📈 Training history saved → /kaggle/working//indicbart_history.csv

SUMMARY
  Model        : ai4bharat/IndicBART
  Best Val PPL : 57.73
  BLEU-1       : 0.9194
  BLEU-2       : 0.8043
  BLEU-4       : 0.5358

GENERATED DOHAS (beam search, num_beams=4)
-------------------------------------------------------
  Theme     : ईश्वर
  Context   : भगवान की शक्ति अपरंपार
  Generated → <s><2hi> भगवन क दखय, भगवन ह भगवन। धरन क धरन ह, धरन दख दन ॥सहत-सहज-सहमत-सहय-सहम-भरम-परम-सहसहज ॥ वरत-सथ-सच कहत-करत ह, कय-कय-सठत-वचत-रत-परत-दख-स-कह-कर-करह-सरत-सरस-सरह-सग-सरव-स
  Reference → साँई ते सब होते है, बन्दे से कुछ नाहिं । राई से पर्वत करे, पर्वत राई माहिं ॥
-------------------------------------------------------
  Theme     : दर्शन
  Context   : त्रिगुणों का बंधन
  Generated → <s><2hi> तरगण क बधन क, तरगण ह तरग। तरग क कय तरग, तरग-तरग क तरग ॥सहत-सहनत-सहमत-सहज-सहम-सतर ॥-वरग-भरम-सहसहसह-सहय-सवरन-परख-सबरग-दखल-ससर-सव-सवन-सखन-सहन-धन-धन क कछ क, कछ-कछ-छव-छड-छह

In [ ]:
CSV_PATH = "/kaggle/input/datasets/kgan31/doha-simple-context/dohas_nlp_ready_simple_lang.csv"
pairs, df = load_dataset(CSV_PATH)

# Verify what's going into the encoder
print("Columns in your CSV:", df.columns.tolist())
print("\nFirst 3 encoder inputs:")
for inp, tgt in pairs[:3]:
    print(f"  INPUT  → {inp}")
    print(f"  TARGET → {tgt[:60]}...")
    print()